# MLP-View lifting, step by step 🧭
### *Ignore the cameras entirely* — the simplest possible view transform, and the ideal negative control

**Lifter name:** `mlp_view`  
**Class:** `MLPViewLifter`  
**Direction:** push (image → BEV), but geometry-free  
**Parameters in lift:** one `Linear(h·w, X·Y)` per camera  
**Paper:** Pan et al., *Cross-View Semantic Segmentation for Sensing Surroundings*, ICCV 2020  

> This notebook is the companion to `lifter_comparison_and_negative_control.ipynb`.  
> Read that one to see how MLP-View compares to *all* the geometric lifters side-by-side.  
> This notebook goes deeper into *why* MLP-View works at all, and exactly where and how it breaks.


## Step 0: Setup and the data

> New to this repo? The notebook `tpv_lifting_step_by_step.ipynb` explains `GridSpec`, `Cameras`, and the synthetic dataset in detail.  
> This notebook assumes you have already run through `simple_bev_lifting_step_by_step.ipynb`.


In [ ]:
import math, time, dataclasses
import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from lifting import GridSpec, Cameras, build_bev_model
from lifting.data import SceneBatch, SyntheticSceneDataset
from lifting.training import BEVSegmentationTask, Trainer
from lifting.lifters import MLPViewLifter

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'image.interpolation': 'nearest'})
print('torch', torch.__version__)

In [ ]:
grid = GridSpec(resolution=(32, 32, 4))   # 32 x 32 x 4 cells, 1 m^3 each, 32 m x 32 m x 4 m region
data = SyntheticSceneDataset(256, grid, seed=1)
batch = SceneBatch.collate([data[i] for i in range(4)])
cams  = batch.cameras
ext   = [grid.x_bounds[0], grid.x_bounds[1], grid.y_bounds[0], grid.y_bounds[1]]
print('images :', tuple(batch.images.shape), '= (B, N cameras, 3, H, W)')
print('BEV labels:', tuple(batch.bev_labels.shape), '= (B, X, Y)')

fig, ax = plt.subplots(1, 5, figsize=(16, 2.6))
for n in range(4):
    ax[n].imshow(batch.images[0, n].permute(1, 2, 0))
    ax[n].set_title(f'camera {n}'); ax[n].axis('off')
ax[4].imshow(batch.bev_labels[0].T, origin='lower', vmin=0, vmax=2, extent=ext)
ax[4].set_title('target: top view'); ax[4].set_xlabel('x forward (m)'); ax[4].set_ylabel('y left (m)')
plt.tight_layout(); plt.show()

In [ ]:
import os, sys
for _p in ('.', 'notebooks', os.path.join('..', 'notebooks')):
    if os.path.exists(os.path.join(_p, 'lifting_diagrams.py')):
        sys.path.insert(0, _p); break
import lifting_diagrams as dg

## Step 1: The big idea — a learned lookup table

Every other lifter in this repo uses the camera's intrinsic matrix **K** and extrinsics **[R|t]** to compute *where* in the image each BEV cell appears. The MLP-View lifter throws all of that away.

**The architecture in one equation:**

For camera $ with feature map  \in \mathbb{R}^{C \times h \times w}$, flatten the spatial cells:

48380f_n = \text{flatten}(F_n) \in \mathbb{R}^{C \times hw}48380

Apply a per-camera linear transform  \in \mathbb{R}^{XY \times hw}$:

48380\text{BEV}_n = W_n \cdot f_n^\top \in \mathbb{R}^{C \times XY}48380

Sum across cameras and reshape to $\mathbb{R}^{C \times X \times Y}$:

48380\text{BEV} = \sum_n \text{BEV}_n48380

That is the **entire lifter**: $ linear layers, one per camera, with no reference to intrinsics or extrinsics anywhere.

The idea is from **View-Parsing-Network (VPN)** — Pan et al., ICCV 2020, *Cross-View Semantic Segmentation for Sensing Surroundings*. They called the mapping a 'view transformer'.


### 🖼️ Diagram: MLP-View vs geometric lifters
Left: geometric lifters use K, R, t at every forward pass to compute pixel coordinates. Right: MLP-View bakes the mapping into weights — if the rig moves, those weights are wrong.


In [ ]:
dg.fig_mlp_vs_geometry()

## Step 2: Instantiate MLPViewLifter — what is actually in it?

The only learnable parameters are the $ linear layers. There are no conv layers, no depth heads, no attention — just a fixed matrix multiply per camera.


In [ ]:
# feature_size = (h, w) of the coarsest feature pyramid level
h, w = 6, 12   # coarsest level at 48x96 image with two 2x poolings
mlp = MLPViewLifter(grid, in_channels=32, out_channels=32,
                    feature_size=(h, w), num_cameras=4)
print('uses_geometry flag:', mlp.uses_geometry, '  (False = this lifter ignores calibration)')
print()
print('Parameters:')
for name, p in mlp.named_parameters():
    print(f'  {name:40s}  {tuple(p.shape)}')
n_lift = sum(p.numel() for p in mlp.parameters())
print(f'\nTotal in lifter: {n_lift:,} parameters')
print(f'Each view_mlp[n]:  {h*w} input cells  x  {grid.num_cells_bev} BEV cells')
print(f'  = one row per BEV cell, one col per image cell')

## Step 3: Anatomy of a forward pass



Critically: **cameras is never touched inside **. You can pass anything as the cameras argument — it is ignored.


In [ ]:
from lifting.encoders import SimpleConvEncoder
enc = SimpleConvEncoder(out_channels=32, num_levels=2)
with torch.no_grad():
    feats = [f.view(4, 4, *f.shape[1:]) for f in enc(batch.images.view(16, 3, 48, 96))]
print('feature levels:', [tuple(f.shape) for f in feats])

# manual forward to trace each step
feat = feats[-1]             # coarsest level, (B, N, C, h, w)
b, n, c, hh, ww = feat.shape
flat = feat.flatten(3)       # (B, N, C, h*w)
print('flat:', tuple(flat.shape))

with torch.no_grad():
    mapped = sum(mlp.view_mlps[i](flat[:, i]) for i in range(n))  # (B, C, X*Y)
    nx, ny = grid.bev_shape
    bev_raw = mapped.view(b, c, nx, ny)
    bev     = mlp.output(bev_raw)
print('BEV raw:', tuple(bev_raw.shape), '-> output:', tuple(bev.shape))

# verify cameras argument is truly unused
with torch.no_grad():
    bev_null_cam = mlp(feats, cameras=None)   # pass None — should not crash
print('Same result with cameras=None:', torch.allclose(bev, bev_null_cam, atol=1e-5))

## Step 4: What does the weight matrix learn?

Each row of $ is the set of image cells that "light up" one BEV cell.  
Before training the weights are random. After training on a fixed rig the rows should look like ray-like streaks from the camera towards the BEV cell — if the learning signal is strong enough.


In [ ]:
# Before training: random weights
W0 = mlp.view_mlps[0].weight.detach()   # (X*Y, h*w)
nx, ny = grid.bev_shape
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(W0.numpy(), aspect='auto', cmap='RdBu_r', vmin=-W0.abs().max(), vmax=W0.abs().max())
axes[0].set_title('weight matrix W[0] (random\nbefore training)')
axes[0].set_xlabel('image cell index (h·w=72)'); axes[0].set_ylabel('BEV cell index (X·Y=1024)')

# Show 4 row slices = 4 different BEV cells
bev_cells = [grid.num_cells_bev // 4 * k for k in range(4)]   # front-left, front-right, back-left, back-right
for ax, idx in zip(axes[1:], bev_cells[:3]):
    row = W0[idx].view(h, w).numpy()
    ax.imshow(row, cmap='RdBu_r', vmin=-abs(row).max(), vmax=abs(row).max())
    bev_x = (idx // ny) * grid.voxel_size[0] + grid.x_bounds[0]
    bev_y = (idx  % ny) * grid.voxel_size[1] + grid.y_bounds[0]
    ax.set_title(f'row for BEV cell\n({bev_x:.0f}m, {bev_y:.0f}m) — random')
    ax.set_xlabel('image col'); ax.set_ylabel('image row')
plt.tight_layout(); plt.show()
print('After training these row-slices should show ray-like patterns (see Step 7)')

## Step 5: Where it works — a camera rig that never moves

If *every* scene is captured with cameras in exactly the same position and orientation, the mapping from image cell to BEV cell is constant. MLP-View can memorise this mapping.

The weight $ learns: "for camera $, image cell $ contributes weight $ to BEV cell $".  
With enough training data, this converges to something like the geometric ray from camera $ through image cell $ — *because the geometry is fixed*.

**Key insight:** MLP-View is not "wrong". It is just *brittle*: it memorises the geometry instead of computing it.


In [ ]:
# Fixed-rig dataset: same camera poses in every scene
torch.manual_seed(0)
data_fix  = SyntheticSceneDataset(256, grid, seed=1, random_rig=False)
data_rand = SyntheticSceneDataset(256, grid, seed=1, random_rig=True)

def make_batches(dataset, lo, hi, bs=4):
    return [SceneBatch.collate([dataset[i + j] for j in range(bs)]) for i in range(lo, hi, bs)]

train_fix,  test_fix  = make_batches(data_fix,  0, 192), make_batches(data_fix,  192, 256)
train_rand, test_rand = make_batches(data_rand, 0, 192), make_batches(data_rand, 192, 256)

def train_mlp(train, test, label):
    model = build_bev_model('mlp_view', grid, num_classes=3, image_size=(48, 96), num_cameras=4,
                            channels=32, feature_size=(6, 12))
    task  = BEVSegmentationTask(model, lr=1e-3)
    trainer = Trainer(task)
    trainer.fit(train, steps=300)
    miou = trainer.evaluate(test).mean_foreground()
    print(f'{label}  mIoU = {miou:.3f}')
    return model, trainer, miou

model_fix, trainer_fix, miou_fix = train_mlp(train_fix, test_fix, 'fixed-rig')

After training on the fixed rig, each row of $ should resemble a **ray pattern** — the image cells that see the corresponding world location.

Compare to the random weights in Step 4.


In [ ]:
W_trained = model_fix.lifter.view_mlps[0].weight.detach()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(W_trained.numpy(), aspect='auto', cmap='RdBu_r',
               vmin=-W_trained.abs().max(), vmax=W_trained.abs().max())
axes[0].set_title('weight matrix W[0]\n(after training, fixed rig)')
axes[0].set_xlabel('image cell index (h·w=72)'); axes[0].set_ylabel('BEV cell index (X·Y=1024)')

for ax, idx in zip(axes[1:], bev_cells[:3]):
    row = W_trained[idx].view(h, w).numpy()
    ax.imshow(row, cmap='RdBu_r', vmin=-abs(row).max(), vmax=abs(row).max())
    bev_x = (idx // ny) * grid.voxel_size[0] + grid.x_bounds[0]
    bev_y = (idx  % ny) * grid.voxel_size[1] + grid.y_bounds[0]
    ax.set_title(f'row for BEV cell\n({bev_x:.0f}m, {bev_y:.0f}m) — trained')
    ax.set_xlabel('image col'); ax.set_ylabel('image row')
plt.tight_layout(); plt.show()

## Step 6: Where it breaks — randomised camera poses

Now train on a dataset where each scene has cameras at a **different random orientation** around the car.  
The image-cell → BEV-cell mapping now changes with every scene. There is no fixed table to memorise.

**Expected outcome:** MLP-View should collapse to a trivial prediction (e.g. "everything is background"),  
while `simple_bev` (which uses the calibration) should maintain its score.


In [ ]:
model_rand, trainer_rand, miou_rand = train_mlp(train_rand, test_rand, 'random-rig')

# also train simple_bev on random-rig for comparison
model_sbev = build_bev_model('simple_bev', grid, num_classes=3, image_size=(48, 96), num_cameras=4, channels=32)
task_sbev  = BEVSegmentationTask(model_sbev, lr=1e-3)
trainer_sbev = Trainer(task_sbev)
trainer_sbev.fit(train_rand, steps=300)
miou_sbev = trainer_sbev.evaluate(test_rand).mean_foreground()
print(f'\nComparison on random-rig:')
print(f'  mlp_view   mIoU = {miou_rand:.3f}  (should be ~0 — no geometry)')
print(f'  simple_bev mIoU = {miou_sbev:.3f}  (should be > 0.15 — uses geometry)')

In [ ]:
# Show prediction vs ground truth for a test scene
tb = test_rand[0]
with torch.no_grad():
    pred_mlp  = trainer_rand.predict(tb)
    pred_sbev = trainer_sbev.predict(tb)

fig, ax = plt.subplots(3, 4, figsize=(12, 8))
titles = ['GT', 'mlp_view\n(random rig)', 'simple_bev\n(random rig)']
preds  = [tb.bev_labels, pred_mlp.argmax(1), pred_sbev.argmax(1)]
for row, (preds_row, title) in enumerate(zip(preds, titles)):
    for s in range(4):
        ax[row, s].imshow(preds_row[s].T, origin='lower', vmin=0, vmax=2, extent=ext)
        ax[row, s].set_title(f'{title} scene {s}' if s == 0 else f'scene {s}')
        ax[row, s].axis('off')
plt.tight_layout(); plt.show()
print('mlp_view collapses to one class; simple_bev maintains structure')

## Step 7: The geometry verification test 🔬

Keep images and labels fixed, but **rotate the camera extrinsics by 90°**. A lifter that truly uses geometry must fail.  
MLP-View **ignores** the passed `cameras` argument, so rotating the rig does nothing — it should score *the same* on correct and rotated extrinsics.

This is exactly the test used by `lifting-verify`. The spec is:
1. Beat MLP-View by ≥ 0.15 mIoU on the random-rig test set.
2. Lose ≥ 50% of the mIoU when extrinsics are rotated 90°.

MLP-View trivially *fails* requirement 2 (it can't lose what it never had).


In [ ]:
Rz = torch.eye(4)
Rz[0,0], Rz[0,1], Rz[1,0], Rz[1,1] = 0., -1., 1., 0.    # 90 degrees about z
rotate_rig = lambda b: dataclasses.replace(b, cameras=b.cameras.transformed(Rz))

# on fixed-rig model
good_fix = trainer_fix.evaluate(test_fix).mean_foreground()
bad_fix  = trainer_fix.evaluate(test_fix, transform=rotate_rig).mean_foreground()

# on random-rig model
good_rand = trainer_rand.evaluate(test_rand).mean_foreground()
bad_rand  = trainer_rand.evaluate(test_rand, transform=rotate_rig).mean_foreground()

print('fixed-rig model:')
print(f'  mIoU with correct extrinsics : {good_fix:.3f}')
print(f'  mIoU with rotated extrinsics : {bad_fix:.3f}')
print(f'  change: {bad_fix - good_fix:+.3f}  (should be ~0 — cameras are ignored)')
print()
print('random-rig model:')
print(f'  mIoU with correct extrinsics : {good_rand:.3f}')
print(f'  mIoU with rotated extrinsics : {bad_rand:.3f}')
print(f'  change: {bad_rand - good_rand:+.3f}  (should be ~0 — cameras are ignored)')

## Step 8: Plug it into a full model

`build_bev_model('mlp_view', ...)` assembles: **image encoder → MLPViewLifter → BEV decoder (U-Net) → segmentation head**.  
The full model is trained end-to-end; the lifter's linear layers receive gradients through the decoder.


In [ ]:
torch.manual_seed(0)
train_batches = make_batches(data_fix, 0, 192)   # use fixed rig for this demo
test_batches  = make_batches(data_fix, 192, 256)

model = build_bev_model('mlp_view', grid, num_classes=3, image_size=(48, 96),
                        num_cameras=4, channels=32, feature_size=(6, 12))
n_enc  = sum(p.numel() for p in model.encoder.parameters())
n_lift = sum(p.numel() for p in model.lifter.parameters())
n_dec  = sum(p.numel() for p in model.decoder.parameters())
print(f'encoder:  {n_enc:>10,} params')
print(f'lifter:   {n_lift:>10,} params  <— the linear tables, {h}×{w}→{grid.num_cells_bev} per camera')
print(f'decoder:  {n_dec:>10,} params')

task    = BEVSegmentationTask(model, lr=1e-3)
trainer = Trainer(task)
trainer.fit(train_batches, steps=300)
miou = trainer.evaluate(test_batches).mean_foreground()
print(f'\nfixed-rig mIoU: {miou:.3f}')

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(trainer.history.losses, alpha=0.3, label='per step')
plt.plot(trainer.history.smoothed(15), label='smoothed')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.tight_layout(); plt.show()

tb = test_batches[0]
pred = trainer.predict(tb)
fig, ax = plt.subplots(2, 4, figsize=(11, 5.6))
for s in range(4):
    ax[0, s].imshow(tb.bev_labels[s].T, origin='lower', vmin=0, vmax=2, extent=ext)
    ax[0, s].set_title(f'GT scene {s}'); ax[0, s].axis('off')
    ax[1, s].imshow(pred[s].argmax(0).T, origin='lower', vmin=0, vmax=2, extent=ext)
    ax[1, s].set_title(f'mlp_view pred {s}'); ax[1, s].axis('off')
plt.tight_layout(); plt.show()

## Step 9: Why do we need a geometry-free baseline at all?

It might seem that MLP-View is simply a bad lifter. But it serves an essential scientific role:

**The two requirements of `lifting-verify`:**

| Requirement | Measures |
| --- | --- |
| Beat MLP-View by ≥ 0.15 mIoU on random-rig data | The lifter actually exploits geometry |
| Lose ≥ 50% of score when extrinsics are rotated 90° | The lifter is *sensitive* to extrinsic changes |

A method that only satisfies requirement 1 but not 2 might have learned a shortcut: it uses the intrinsics but not the extrinsics, or it found some other geometry-free pattern that generalises.  
Both conditions together provide a strong signal that the correct 3D geometry is being exploited.

A method that satisfies *neither* (like MLP-View on random rigs) is just the lower bound: it tells us what "no geometry" looks like, so we can measure how much any geometric lifter gains over it.


In [ ]:
# Quick comparison: MLP-View vs geometric lifters
# (run lifter_comparison_and_negative_control.ipynb for the full shoot-out)
results = [
    ('mlp_view',   miou_fix,  miou_rand, False),
    ('simple_bev', None,      miou_sbev, True),
]
print(f"{'lifter':12s}  {'fixed-rig':>10s}  {'random-rig':>11s}  uses geometry")
for name, m_fix, m_rand, geo in results:
    m_fix_str  = f'{m_fix:.3f}' if m_fix  is not None else '     -'
    m_rand_str = f'{m_rand:.3f}' if m_rand is not None else '     -'
    print(f'{name:12s}  {m_fix_str:>10s}  {m_rand_str:>11s}  {geo}')

### 🖼️ Diagram: the whole MLP-View pipeline


In [ ]:
dg.pipeline([
    ('Image encoder', 'CNN\n(B,N,C,H,W)'),
    ('Flatten spatial', '(B,N,C,h·w)'),
    ('Per-camera Linear', 'W_n: h·w → X·Y\n(one per camera)'),
    ('Sum over cameras', '(B,C,X·Y)'),
    ('Conv+BN+ReLU', '(B,out_C,X,Y)'),
    ('BEV decoder', '(B,classes,X,Y)'),
], purple=['Per-camera Linear', 'Conv+BN+ReLU'], title='MLP-View pipeline')

## Step 10: Recap, pros, cons, and exercises



**Pros:**
- Fewest parameters and FLOPs of any lifter in this repo.
- Works well when the camera rig is fixed — competitive with Simple-BEV on fixed-rig benchmarks.
- Essential as a negative control in the `lifting-verify` suite.

**Cons:**
- Cannot generalise to unseen camera poses — the weights *are* the geometry.
- Every pixel feature is smeared over the entire BEV plane (no depth).
- The lifting capacity scales as (h \cdot w \cdot X \cdot Y \cdot N)$ — expensive for large images or high-resolution BEV.

**Exercises:**
1. Replace `nn.Linear` with a two-layer MLP (e.g. `h·w → 256 → X·Y`). Does it generalise better to random rigs?
2. Visualise what the trained weight rows look like for a scene with very good mIoU vs one with very bad mIoU.
3. Pass the intrinsic matrix as extra input alongside the flattened features. Does it learn to use it?

**References:**
- Pan et al., *Cross-View Semantic Segmentation for Sensing Surroundings*, ICCV 2020. arXiv:2005.07099
- Harley et al., *Simple-BEV: What Really Matters for Multi-Sensor BEV Perception?*, IROS 2023. arXiv:2206.07959 (contains the MLP-View baseline in Table 5)
- This repo's `lifter_comparison_and_negative_control.ipynb` — the full 7-lifter shoot-out.
